%sql
-- Fresh rerun cleanup for Epic Clarity person
-- TRUNCATE TABLE _exponent.omop_epic.person;

DELETE FROM _exponent.omop_silver.person
WHERE source_system = 'epic_clarity';

DELETE FROM _exponent.omop_mapping.source_to_person
WHERE source_system = 'epic_clarity';

In [0]:
source = 'epic_clarity'

In [ ]:
%sql
-- Insert concept mappings for Epic Clarity demographic codes
-- NOTE: ZC_PATIENT_RACE and ZC_ETHNIC_GROUP reference tables not available in bronze layer
-- Mappings based on typical Epic coded values

-- GENDER mappings (SEX_C): 1=Female, 2=Male (CORRECTED - verified via patient name analysis)
-- Previous mapping was backwards!
INSERT INTO _exponent.omop_mapping.domain_source_to_concept (
  source_system, source_table, source_field, domain_id, source_id, source_value, omop_concept_id, active_flag, last_update_tsp
)
SELECT * FROM (
  VALUES
    ('epic_clarity', 'patient', 'SEX_C', 'Gender', '1', 'Female', 8532, TRUE, CURRENT_TIMESTAMP()),
    ('epic_clarity', 'patient', 'SEX_C', 'Gender', '2', 'Male', 8507, TRUE, CURRENT_TIMESTAMP()),
    ('epic_clarity', 'patient', 'SEX_C', 'Gender', '3', 'Unknown', 8551, TRUE, CURRENT_TIMESTAMP()),
    ('epic_clarity', 'patient', 'SEX_C', 'Gender', '950', 'Unknown', 8551, TRUE, CURRENT_TIMESTAMP()),
    ('epic_clarity', 'patient', 'SEX_C', 'Gender', '951', 'Unknown', 8551, TRUE, CURRENT_TIMESTAMP()),
    ('epic_clarity', 'patient', 'SEX_C', 'Gender', '', 'Unknown', 8551, TRUE, CURRENT_TIMESTAMP())
) AS v(source_system, source_table, source_field, domain_id, source_id, source_value, omop_concept_id, active_flag, last_update_tsp)
WHERE NOT EXISTS (
  SELECT 1 FROM _exponent.omop_mapping.domain_source_to_concept d
  WHERE d.source_system = v.source_system
    AND d.domain_id = v.domain_id
    AND d.source_id = v.source_id
);

-- RACE mappings (PATIENT_RACE_C): Based on typical Epic ZC_PATIENT_RACE codes
INSERT INTO _exponent.omop_mapping.domain_source_to_concept (
  source_system, source_table, source_field, domain_id, source_id, source_value, omop_concept_id, active_flag, last_update_tsp
)
SELECT * FROM (
  VALUES
    ('epic_clarity', 'patient_race', 'PATIENT_RACE_C', 'Race', '1', 'White', 8527, TRUE, CURRENT_TIMESTAMP()),
    ('epic_clarity', 'patient_race', 'PATIENT_RACE_C', 'Race', '2', 'Black or African American', 8516, TRUE, CURRENT_TIMESTAMP()),
    ('epic_clarity', 'patient_race', 'PATIENT_RACE_C', 'Race', '3', 'Asian', 8515, TRUE, CURRENT_TIMESTAMP()),
    ('epic_clarity', 'patient_race', 'PATIENT_RACE_C', 'Race', '6', 'Unknown/Declined', 0, TRUE, CURRENT_TIMESTAMP()),
    ('epic_clarity', 'patient_race', 'PATIENT_RACE_C', 'Race', '7', 'American Indian or Alaska Native', 8657, TRUE, CURRENT_TIMESTAMP()),
    ('epic_clarity', 'patient_race', 'PATIENT_RACE_C', 'Race', '8', 'Other', 0, TRUE, CURRENT_TIMESTAMP()),
    ('epic_clarity', 'patient_race', 'PATIENT_RACE_C', 'Race', '14', 'Native Hawaiian or Pacific Islander', 8557, TRUE, CURRENT_TIMESTAMP()),
    ('epic_clarity', 'patient_race', 'PATIENT_RACE_C', 'Race', '25', 'Multiple Races', 0, TRUE, CURRENT_TIMESTAMP())
) AS v(source_system, source_table, source_field, domain_id, source_id, source_value, omop_concept_id, active_flag, last_update_tsp)
WHERE NOT EXISTS (
  SELECT 1 FROM _exponent.omop_mapping.domain_source_to_concept d
  WHERE d.source_system = v.source_system
    AND d.domain_id = v.domain_id
    AND d.source_id = v.source_id
);

-- ETHNICITY mappings (ETHNIC_GROUP_C): Based on typical Epic ZC_ETHNIC_GROUP codes
INSERT INTO _exponent.omop_mapping.domain_source_to_concept (
  source_system, source_table, source_field, domain_id, source_id, source_value, omop_concept_id, active_flag, last_update_tsp
)
SELECT * FROM (
  VALUES
    ('epic_clarity', 'patient', 'ETHNIC_GROUP_C', 'Ethnicity', '1', 'Not Hispanic or Latino', 38003564, TRUE, CURRENT_TIMESTAMP()),
    ('epic_clarity', 'patient', 'ETHNIC_GROUP_C', 'Ethnicity', '3', 'Hispanic or Latino', 38003563, TRUE, CURRENT_TIMESTAMP()),
    ('epic_clarity', 'patient', 'ETHNIC_GROUP_C', 'Ethnicity', '4', 'Unknown', 0, TRUE, CURRENT_TIMESTAMP()),
    ('epic_clarity', 'patient', 'ETHNIC_GROUP_C', 'Ethnicity', '9', 'Declined', 0, TRUE, CURRENT_TIMESTAMP())
) AS v(source_system, source_table, source_field, domain_id, source_id, source_value, omop_concept_id, active_flag, last_update_tsp)
WHERE NOT EXISTS (
  SELECT 1 FROM _exponent.omop_mapping.domain_source_to_concept d
  WHERE d.source_system = v.source_system
    AND d.domain_id = v.domain_id
    AND d.source_id = v.source_id
)

In [ ]:
silver_person_df = spark.sql(f'''
SELECT
    CONCAT_WS(CHR(31), 'epic_clarity', 'PATIENT', 'PAT_ID', p.PAT_ID) AS person_source_value,
    p.BIRTH_DATE AS birth_datetime,
    YEAR(p.BIRTH_DATE) AS year_of_birth,
    MONTH(p.BIRTH_DATE) AS month_of_birth,
    DAY(p.BIRTH_DATE) AS day_of_birth,
    0 AS gender_concept_id,
    0 AS race_concept_id,
    0 AS ethnicity_concept_id,
    COALESCE(CAST(p.SEX_C AS STRING), '') AS gender_source_value,
    0 AS gender_source_concept_id,
    COALESCE(CAST(pr.PATIENT_RACE_C AS STRING), '') AS race_source_value,
    0 AS race_source_concept_id,
    COALESCE(CAST(p.ETHNIC_GROUP_C AS STRING), '') AS ethnicity_source_value,
    0 AS ethnicity_source_concept_id,
    'epic_clarity' AS source_system,
    CURRENT_TIMESTAMP() AS updated_tsp
FROM _exponent._bronze_epic_clarity.patient p
LEFT JOIN _exponent._bronze_epic_clarity.patient_race pr ON p.PAT_ID = pr.PAT_ID
WHERE p.PAT_ID IS NOT NULL
  AND p.BIRTH_DATE IS NOT NULL
  -- Exclude implausible birth dates (before 1900)
  AND YEAR(p.BIRTH_DATE) >= 1900
  -- Exclude patients that have been merged into other patients
  AND NOT EXISTS (
    SELECT 1 FROM _exponent._bronze_epic_clarity.pat_merge_history pmh
    WHERE pmh.PAT_ID = p.PAT_ID
      AND (pmh.DELETE_FLAG = 0 OR pmh.DELETE_FLAG IS NULL)
  )
''')

display(silver_person_df)
silver_person_df.createOrReplaceTempView("silver_person")

Added Row Number func to Merge to stop dupicate errors

In [0]:
%sql
MERGE INTO _exponent.omop_silver.person AS t
USING (
  SELECT * FROM (
    SELECT *,
      ROW_NUMBER() OVER (
        PARTITION BY person_source_value
        ORDER BY person_source_value
      ) AS rn
    FROM silver_person
  ) WHERE rn = 1
) AS s
ON t.person_source_value = s.person_source_value

WHEN MATCHED AND (
     NOT (t.year_of_birth <=> s.year_of_birth)
  OR NOT (t.month_of_birth <=> s.month_of_birth)
  OR NOT (t.day_of_birth <=> s.day_of_birth)
  OR NOT (t.birth_datetime <=> s.birth_datetime)
  OR NOT (t.gender_concept_id <=> s.gender_concept_id)
  OR NOT (t.race_concept_id <=> s.race_concept_id)
  OR NOT (t.ethnicity_concept_id <=> s.ethnicity_concept_id)
  OR NOT (t.gender_source_value <=> s.gender_source_value)
  OR NOT (t.gender_source_concept_id <=> s.gender_source_concept_id)
  OR NOT (t.race_source_value <=> s.race_source_value)
  OR NOT (t.race_source_concept_id <=> s.race_source_concept_id)
  OR NOT (t.ethnicity_source_value <=> s.ethnicity_source_value)
  OR NOT (t.ethnicity_source_concept_id <=> s.ethnicity_source_concept_id)
)
THEN UPDATE SET
  t.birth_datetime = s.birth_datetime,
  t.year_of_birth = s.year_of_birth,
  t.month_of_birth = s.month_of_birth,
  t.day_of_birth = s.day_of_birth,
  t.gender_concept_id = s.gender_concept_id,
  t.race_concept_id = s.race_concept_id,
  t.ethnicity_concept_id = s.ethnicity_concept_id,
  t.gender_source_value = s.gender_source_value,
  t.gender_source_concept_id = s.gender_source_concept_id,
  t.race_source_value = s.race_source_value,
  t.race_source_concept_id = s.race_source_concept_id,
  t.ethnicity_source_value = s.ethnicity_source_value,
  t.ethnicity_source_concept_id = s.ethnicity_source_concept_id,
  t.last_mod_tsp = s.updated_tsp

WHEN NOT MATCHED THEN
INSERT (
  person_source_value,
  birth_datetime,
  year_of_birth,
  month_of_birth,
  day_of_birth,
  gender_concept_id,
  race_concept_id,
  ethnicity_concept_id,
  gender_source_value,
  gender_source_concept_id,
  race_source_value,
  race_source_concept_id,
  ethnicity_source_value,
  ethnicity_source_concept_id,
  source_system,
  last_mod_tsp
)
VALUES (
  s.person_source_value,
  s.birth_datetime,
  s.year_of_birth,
  s.month_of_birth,
  s.day_of_birth,
  s.gender_concept_id,
  s.race_concept_id,
  s.ethnicity_concept_id,
  s.gender_source_value,
  s.gender_source_concept_id,
  s.race_source_value,
  s.race_source_concept_id,
  s.ethnicity_source_value,
  s.ethnicity_source_concept_id,
  s.source_system,
  s.updated_tsp
)

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_person (
    source_system,
    person_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    s.source_system,
    s.person_source_value,
    TRUE                AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    CURRENT_TIMESTAMP() AS last_mod_tsp,
    NULL                AS merge_id,
    NULL                AS merge_reason
FROM (
    SELECT DISTINCT source_system, person_source_value
    FROM _exponent.omop_silver.person
    WHERE person_source_value IS NOT NULL
      AND source_system = 'epic_clarity'
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_person x
  ON s.person_source_value = x.person_source_value
 AND x.source_system = 'epic_clarity';

In [0]:
# %sql
# select *
# from _exponent.omop_mapping.source_to_person
# where source_system like '%epic%'
# limit 100;

In [ ]:
gold_person_df = spark.sql("""
SELECT
    source_to_person.person_id,
    s.birth_datetime,
    s.year_of_birth,
    s.month_of_birth,
    s.day_of_birth,
    -- Default to 8551 (Unknown) instead of 0 for unmapped gender
    COALESCE(gc.omop_concept_id, 8551) AS gender_concept_id,
    COALESCE(rc.omop_concept_id, 0) AS race_concept_id,
    COALESCE(ec.omop_concept_id, 0) AS ethnicity_concept_id,
    s.gender_source_value,
    0                               AS gender_source_concept_id,
    s.race_source_value,
    0                               AS race_source_concept_id,
    s.ethnicity_source_value,
    0                               AS ethnicity_source_concept_id,
    mp.provider_id,
    mc.care_site_id,
    ml.location_id,
    s.last_mod_tsp AS updated_tsp
FROM _exponent.omop_silver.person s
INNER JOIN _exponent.omop_mapping.source_to_person source_to_person
    ON s.person_source_value = source_to_person.person_source_value
    AND source_to_person.source_system = 'epic_clarity'
    AND source_to_person.active_flag = TRUE
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept gc
    ON gc.source_id = s.gender_source_value
    AND gc.domain_id = 'Gender'
    AND gc.source_system = 'epic_clarity'
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept rc
    ON rc.source_id = s.race_source_value
    AND rc.domain_id = 'Race'
    AND rc.source_system = 'epic_clarity'
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept ec
    ON ec.source_id = s.ethnicity_source_value
    AND ec.domain_id = 'Ethnicity'
    AND ec.source_system = 'epic_clarity'
LEFT JOIN _exponent._bronze_epic_clarity.patient p
    ON CONCAT_WS(CHR(31), 'epic_clarity', 'PATIENT', 'PAT_ID', p.PAT_ID) = s.person_source_value
LEFT JOIN _exponent.omop_mapping.source_to_provider mp
    ON CONCAT_WS(CHR(31), 'epic_clarity', 'CLARITY_SER', 'PROV_ID', p.CUR_PCP_PROV_ID) = mp.provider_source_value
    AND mp.source_system = 'epic_clarity'
    AND mp.active_flag = TRUE
LEFT JOIN _exponent.omop_mapping.source_to_care_site mc
    ON CONCAT_WS(CHR(31), 'epic_clarity', 'CLARITY_DEP', 'DEPARTMENT_ID', CAST(p.CUR_PRIM_LOC_ID AS STRING)) = mc.care_site_source_value
    AND mc.source_system = 'epic_clarity'
    AND mc.active_flag = TRUE
LEFT JOIN _exponent.omop_mapping.source_to_location ml
    ON s.person_source_value = ml.location_source_value
    AND ml.source_system = 'epic_clarity'
    AND ml.active_flag = TRUE
WHERE s.source_system = 'epic_clarity'
""")

display(gold_person_df)
gold_person_df.createOrReplaceTempView("gold_person")

In [0]:
%sql
-- MERGE INTO _exponent.omop.person AS gold_person
MERGE INTO _exponent.omop_epic.person AS gold_person
USING gold_person AS src
ON gold_person.person_id = src.person_id

WHEN MATCHED AND (
     NOT (gold_person.gender_concept_id <=> src.gender_concept_id)
  OR NOT (gold_person.year_of_birth <=> src.year_of_birth)
  OR NOT (gold_person.month_of_birth <=> src.month_of_birth)
  OR NOT (gold_person.day_of_birth <=> src.day_of_birth)
  OR NOT (gold_person.birth_datetime <=> src.birth_datetime)
  OR NOT (gold_person.race_concept_id <=> src.race_concept_id)
  OR NOT (gold_person.ethnicity_concept_id <=> src.ethnicity_concept_id)
  OR NOT (gold_person.provider_id <=> src.provider_id)
  OR NOT (gold_person.care_site_id <=> src.care_site_id)
  OR NOT (gold_person.location_id <=> src.location_id)
)
THEN UPDATE SET
  gold_person.birth_datetime = src.birth_datetime,
  gold_person.year_of_birth = src.year_of_birth,
  gold_person.month_of_birth = src.month_of_birth,
  gold_person.day_of_birth = src.day_of_birth,
  gold_person.gender_concept_id = src.gender_concept_id,
  gold_person.race_concept_id = src.race_concept_id,
  gold_person.ethnicity_concept_id = src.ethnicity_concept_id,
  gold_person.provider_id = src.provider_id,
  gold_person.care_site_id = src.care_site_id,
  gold_person.location_id = src.location_id

WHEN NOT MATCHED THEN INSERT (
  person_id,
  birth_datetime,
  year_of_birth,
  month_of_birth,
  day_of_birth,
  gender_concept_id,
  race_concept_id,
  ethnicity_concept_id,
  gender_source_value,
  gender_source_concept_id,
  race_source_value,
  race_source_concept_id,
  ethnicity_source_value,
  ethnicity_source_concept_id,
  provider_id,
  care_site_id,
  location_id
)
VALUES (
  src.person_id,
  src.birth_datetime,
  src.year_of_birth,
  src.month_of_birth,
  src.day_of_birth,
  src.gender_concept_id,
  src.race_concept_id,
  src.ethnicity_concept_id,
  src.gender_source_value,
  src.gender_source_concept_id,
  src.race_source_value,
  src.race_source_concept_id,
  src.ethnicity_source_value,
  src.ethnicity_source_concept_id,
  src.provider_id,
  src.care_site_id,
  src.location_id
)

In [ ]:
%sql
-- Validation: Person counts across layers
SELECT 'Bronze (source)' AS layer, COUNT(*) AS person_count 
FROM _exponent._bronze_epic_clarity.patient 
WHERE PAT_ID IS NOT NULL AND BIRTH_DATE IS NOT NULL
UNION ALL
SELECT 'Silver' AS layer, COUNT(*) AS person_count 
FROM _exponent.omop_silver.person WHERE source_system = 'epic_clarity'
UNION ALL
SELECT 'Mapping' AS layer, COUNT(*) AS person_count 
FROM _exponent.omop_mapping.source_to_person WHERE source_system = 'epic_clarity'
UNION ALL
SELECT 'Gold' AS layer, COUNT(*) AS person_count 
FROM _exponent.omop_epic.person